# Matrix multiplication from foundations

## Load dependencies

In [ ]:
# !pip install -q matplotlib
# !pip install torch
# !pip install numba

In [ ]:
from pathlib import Path
import pickle, gzip, math, os, time, shutil
import matplotlib as mpl, matplotlib.pyplot as plt

## Get data

In [ ]:
MNIST_URL='https://github.com/mnielsen/neural-networks-and-deep-learning/blob/master/data/mnist.pkl.gz?raw=true'
path_data = Path('data')
path_data.mkdir(exist_ok=True)
path_gz = path_data/'mnist.pkl.gz'

In [ ]:
from urllib.request import urlretrieve
if not path_gz.exists(): urlretrieve(MNIST_URL, path_gz)

In [ ]:
!ls -l data

In [ ]:
with gzip.open(path_gz, 'rb') as f: ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')

## Convert data to Image matrix

In [ ]:
def chunks(x, sz):
    for i in range(0, len(x), sz): yield x[i:i+sz]

In [ ]:
mpl.rcParams['image.cmap'] = 'gray'
lst1 = list(x_train[0])
plt.imshow(list(chunks(lst1, 28)));

In [ ]:
from itertools import islice

In [ ]:
it = iter(lst1)
img = list(iter(lambda: list(islice(it, 28)), []))
plt.imshow(img);

## Matrix and tensor

In [ ]:
class Matrix:
    def __init__(self, xs): self.xs = xs
    def __getitem__(self, idxs): return self.xs[idxs[0]][idxs[1]]

In [ ]:
m = Matrix(img)
m[20,15]

In [ ]:
import torch
from torch import tensor

In [ ]:
with gzip.open(path_gz, 'rb') as f: ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')
x_train,y_train,x_valid,y_valid = map(tensor, (x_train,y_train,x_valid,y_valid))
print(x_train.shape)
print(x_train.type())
imgs = x_train.reshape((-1,28,28))
print(imgs.shape)
print(imgs[0,20,15])

In [ ]:
plt.imshow(imgs[0]);

In [ ]:
n,c = x_train.shape
y_train, y_train.shape
print(min(y_train),max(y_train))
print(y_train.min(), y_train.max())

## Random numbers

Based on the Wichmann Hill algorithm used before Python 2.3.

In [ ]:
rnd_state = None
def seed(a):
    global rnd_state
    a, x = divmod(a, 30268)
    a, y = divmod(a, 30306)
    a, z = divmod(a, 30322)
    rnd_state = int(x)+1, int(y)+1, int(z)+1

In [ ]:
seed(457428938475)
rnd_state

In [ ]:
def rand():
    global rnd_state
    x, y, z = rnd_state
    x = (171 * x) % 30269
    y = (172 * y) % 30307
    z = (170 * z) % 30323
    rnd_state = x,y,z
    return (x/30269 + y/30307 + z/30323) % 1.0

In [ ]:
rand(),rand(),rand()

In [ ]:
fig, axs = plt.subplots(1,2, figsize = (10,4))
axs[0].plot([rand() for _ in range(50)]);
axs[1].hist([rand() for _ in range(10000)]);

In [ ]:
%timeit -n 10 list(chunks([rand() for _ in range(7840)], 10))
%timeit -n 10 torch.randn(784,10)

### Hnadling random number generation in multiprocess/parallel computing
If a methods produces equal random numbers in two procs:
- It can't handle random number generation in parallel computing
- It will result in wrong calculations

In [ ]:
if os.fork(): print(f'In parent: {rand()}')
else:
    print(f'In child: {rand()}')
    os._exit(os.EX_OK)

In [ ]:
if os.fork(): print(f'In parent: {torch.rand(1)}')
else:
    print(f'In child: {torch.rand(1)}')
    os._exit(os.EX_OK)

In [ ]:
from numpy.random import randn
if os.fork(): print(f'In parent: {randn(1)}')
else:
    print(f'In child: {randn(1)}')
    os._exit(os.EX_OK)

In [ ]:
from random import random
if os.fork(): print(f'In parent: {random()}')
else:
    print(f'In child: {random()}')
    os._exit(os.EX_OK)

## Matrix multiplication

In [ ]:
# torch and numpy print options
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
import numpy as np
np.set_printoptions(precision=2, linewidth=140)

In [ ]:
torch.manual_seed(1)
weights = torch.randn(784,10)
bias = torch.zeros(10)

m1 = x_valid[:5]
m2 = weights

In [ ]:
def matmul(a,b):
    (ar,ac),(br,bc) = a.shape,b.shape
    # print(f"(ar,ac) = {(ar,ac)} | (br,bc) = {(br,bc)}")
    # print(f"total multiplications: {ar*bc*ac}")
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc):
            for k in range(ac): c[i,j] += a[i,k] * b[k,j]
    return c

t1 = matmul(m1, m2)

In [ ]:
%timeit -n 5 _ = matmul(m1, m2)

### Numba

In [ ]:
from numba import njit
from numpy import array

In [ ]:
# from fastcore.test import test_close
# https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/test.py
from functools import partial
from typing import Iterable,Generator
def is_close(a,b,eps=1e-5):
    "Is `a` within `eps` of `b`"
    if hasattr(a, '__array__') or hasattr(b,'__array__'):
        return (abs(a-b)<eps).all()
    if isinstance(a, (Iterable,Generator)) or isinstance(b, (Iterable,Generator)):
        return all(abs(a_-b_)<eps for a_,b_ in zip(a,b))
    return abs(a-b)<eps

def test(a, b, cmp, cname=None):
    "`assert` that `cmp(a,b)`; display inputs and `cname or cmp.__name__` if it fails"
    if cname is None: cname=cmp.__name__
    assert cmp(a,b),f"{cname}:\n{a}\n{b}"

def test_close(a,b,eps=1e-5):
    "`test` that `a` is within `eps` of `b`"
    test(a,b,partial(is_close,eps=eps),'close')

In [ ]:
@njit
def dot(a,b):
    res = 0.
    for i in range(len(a)): res+=a[i]*b[i]
    return res
# the first run is slower because it includes the complation time.

def matmul(a,b):
    (ar,ac),(br,bc) = a.shape,b.shape
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc): c[i,j] = dot(a[i,:], b[:,j])
    # Now only two of our loops are running in Python, not three:
    return c

In [ ]:
m1a,m2a = m1.numpy(),m2.numpy() # numba only works with numpy
test_close(t1,matmul(m1a, m2a))

In [ ]:
%timeit -n 50 t2 = matmul(m1a,m2a)
# about 2000 times fater

### Elementwise ops

In [ ]:
a = tensor([10., 6, -4])
b = tensor([2., 8, 7])
print(a,b)
print(a + b)
print((a < b).float().mean())

In [ ]:
def matmul(a,b):
    (ar,ac),(br,bc) = a.shape,b.shape
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc): c[i,j] = (a[i,:] * b[:,j]).sum()
    return c

test_close(t1,matmul(m1, m2))

In [ ]:
%timeit -n 50 _=matmul(m1, m2)
# about 1000 times faster than the baseline
# about 2 times slower than the numba version!!!

In [ ]:
def matmul(a,b):
    (ar,ac),(br,bc) = a.shape,b.shape
    c = torch.zeros(ar, bc)
    for i in range(ar):
        for j in range(bc): c[i,j] = torch.dot(a[i,:], b[:,j])
    return c

test_close(t1,matmul(m1, m2))

In [ ]:
%timeit -n 50 _=matmul(m1, m2)

#### Frobenius norm

$$\| A \|_F = \left( \sum_{i,j=1}^n | a_{ij} |^2 \right)^{1/2}$$

*Hint*:
- you don't normally need to write equations in LaTeX yourself
- you can click 'edit' in Wikipedia and copy the LaTeX from there.
- Or on arxiv.org,
    - click "Download: Other formats" in the top right, then "Download source";
        - rename the downloaded file to end in `.tgz` if it doesn't already
    - you should find the source there, including the equations to copy and paste.
    - This is the source LaTeX for the equation above:
        ```latex
        $$\| A \|_F = \left( \sum_{i,j=1}^n | a_{ij} |^2 \right)^{1/2}$$
        ```

In [ ]:
m = tensor([[1., 2, 3], [4,5,6], [7,8,9]])
print(m)
sf = (m*m).sum()
print(f"sf = {sf}")
print(f"sf.sqrt() = {sf.sqrt()}")

## Broadcasting

The term **broadcasting** describes how arrays with different shapes are treated during arithmetic operations.

From the [Numpy Documentation](https://docs.scipy.org/doc/numpy-1.10.0/user/basics.broadcasting.html):

    The term broadcasting describes how numpy treats arrays with
    different shapes during arithmetic operations. Subject to certain
    constraints, the smaller array is “broadcast” across the larger
    array so that they have compatible shapes. Broadcasting provides a
    means of vectorizing array operations so that looping occurs in C
    instead of Python. It does this without making needless copies of
    data and usually leads to efficient algorithm implementations.
    
In addition to the efficiency of broadcasting, it allows developers to write less code, which typically leads to fewer errors.

*This section was adapted from [Chapter 4](http://nbviewer.jupyter.org/github/fastai/numerical-linear-algebra/blob/master/nbs/4.%20Compressed%20Sensing%20of%20CT%20Scans%20with%20Robust%20Regression.ipynb#4.-Compressed-Sensing-of-CT-Scans-with-Robust-Regression) of the fast.ai [Computational Linear Algebra](https://github.com/fastai/numerical-linear-algebra) course.*

### Broadcasting with a scalar

In [ ]:
print(a > 0)
print(a + 1)
print(2*m)

### Broadcasting a vector to a matrix

Although broadcasting a scalar is an idea that dates back to APL, the more powerful idea of broadcasting across higher rank tensors [comes from](https://mail.python.org/pipermail/matrix-sig/1995-November/000143.html) a little known language called [Yorick](https://software.llnl.gov/yorick-doc/manual/yorick_50.html).

We can also broadcast a vector to a matrix:

In [ ]:
c = tensor([10.,20,30])
print(c)
print(f"m.shape: {m.shape} | c.shape: {c.shape}")
print(m + c)
print(c + m)

In [ ]:
t = c.expand_as(m)
# the expansion works from right to left.
# m      (2d array):  3 x 3
# c      (1d array):      3 --> 1 x 3 --> 3 x 3
print(t)
print(m + t)

We don't really copy the rows, but it looks as if we did. In fact, the rows are given a *stride* of 0.

In [ ]:
print(f"t.storage():\n{t.storage()})")
print(f"t.stride():\n{t.stride()})")
print(f"t.shape:\n{t.shape})")

### Unsqueeze

You can index with the special value [None] or use `unsqueeze()` to convert a 1-dimensional array into a 2-dimensional array (although one of those dimensions has value 1).

In [ ]:
print(f"c.unsqueeze(0)\n {c.unsqueeze(0)}")
print(f"c[None, :]\n {c[None, :]}")
print(f"c[None]\n {c[None]}")

print(f"c.shape: {c.shape}")
print(f"c.unsqueeze(0).shape: {c.unsqueeze(0).shape}")

In [ ]:
print(f"c.unsqueeze(1):\n{c.unsqueeze(1)}")
print(f"c[:, None]:\n{c[:, None]}")
print(f"c.shape: {c.shape} | c.unsqueeze(1).shape: {c.unsqueeze(1).shape}")

You can always skip trailling ':'s. And '...' means '*all preceding dimensions*'

In [ ]:
print(f"m.shape: {m.shape}")
print(f"m[None].shape: {m[None].shape}")
print(f"m[None, :].shape: {m[None].shape}")
print(f"m[None, ...].shape: {m[None].shape}")
print(f"m[..., None].shape: {m[..., None].shape}")
print(f"m.unsqueeze(-1).shape: {m.unsqueeze(-1).shape}")
print(f"m[:, None].shape: {m[:, None].shape}")
print(f"m.unsqueeze(1).shape: {m.unsqueeze(1).shape}")

In [ ]:
print(c[:,None].expand_as(m))
# m      (2d array):  3 x 3
# c      (2d array):  3 x 1 --> --> 3 x 3
print(m + c[:,None])
print(m + c[None,:])

### Broadcasting Rules

In [ ]:
print(c[None,:] * c[:,None])
# A      (2d array):  1 x 3 --> 3 x 3
# B      (2d array):  3 x 1 --> 3 x 3
# Result (2d array):  3 x 3
print(c[None] > c[:,None])

- When operating on two arrays/tensors, Numpy/PyTorch compares their shapes element-wise.
- It starts with the **trailing dimensions**, and works its way forward. Two dimensions are **compatible** when
    - they are equal, or
    - one of them is 1, in which case that dimension is broadcasted to make it the same size

- Arrays do not need to have the same number of dimensions.
- For example, if you have a `256*256*3` array of RGB values, and you want to scale each color in the image by a different value,
- you can multiply the image by a one-dimensional array with 3 values.
- Lining up the sizes of the trailing axes of these arrays according to the broadcast rules, shows that they are compatible:
```
Image  (3d array): 256 x 256 x 3
Scale  (1d array):             3
Result (3d array): 256 x 256 x 3
```

The [numpy documentation](https://docs.scipy.org/doc/numpy-1.13.0/user/basics.broadcasting.html#general-broadcasting-rules) includes several examples of what dimensions can and can not be broadcast together.

## Matmul with broadcasting

In [ ]:
digit = m1[0]
print(m2.shape)
print(digit.shape)
print(digit[:,None].shape)
print(digit[:,None].expand_as(m2).shape)
print((digit[:,None]*m2).shape)

In [ ]:
def matmul(a,b):
    (ar,ac),(br,bc) = a.shape,b.shape
    c = torch.zeros(ar, bc)
    for i in range(ar):
#       c[i,j] = (a[i,:] * b[:,j]).sum()      # previous version
        c[i]   = (a[i,:,None] * b).sum(dim=0) # broadcast version
    return c

test_close(t1,matmul(m1, m2))

In [ ]:
%timeit -n 50 _=matmul(m1, m2)
# about 5k times faster than the basline (fastest so far)

Our time has gone from ~800ms to ~0.16ms, an over 5000x improvement! We can run on the whole dataset now.

In [ ]:
%timeit _ = matmul(x_train, weights)

In [ ]:
tr = matmul(x_train, weights)
print(tr.shape)
print(tr)

## Einstein summation

[Einstein summation](https://ajcr.net/Basic-guide-to-einsum/) ([`einsum`](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)) is a compact representation for combining products and sums in a general way.

In [ ]:
print(f"m1.shape: {m1.shape} | m2.shape: {m2.shape}")

### Einsum key rules 1
- Repeating letters between input arrays means that values along those axes will be multiplied together.

In [ ]:
# c[i,j] += a[i,k] * b[k,j]
# c[i,j] = (a[i,:] * b[:,j]).sum()
mr = torch.einsum('ik,kj->ikj', m1, m2)
print(mr.shape)
print(mr.sum(1))

### Einsum key rules 2
- Omitting a letter from the output means that values along that axis will be summed.

**Note**: there are much more to it (see the docs linked above)

In [ ]:
torch.einsum('ik,kj->ij', m1, m2)

In [ ]:
def matmul(a,b): return torch.einsum('ik,kj->ij', a, b)
test_close(tr, matmul(x_train, weights), eps=1e-3)

In [ ]:
%timeit -n 50 _=matmul(x_train, weights)

- About 70x faster than the previous fastest implementation
- About 350000x faster than the baseline

### Pytorch op

We can use pytorch's function or operator directly for matrix multiplication.
- the time is very close to the `einsum` implementation since it is used by pytorch

In [ ]:
test_close(tr, x_train@weights, eps=1e-3)

In [ ]:
%timeit -n 50 _=torch.matmul(x_train, weights)

## CUDA

To benefit from GPU-based operation
- We should defines operation that can be run in parallel (dependent from each other)
- The concept of `grid` serves that purpose

In [ ]:
def matmul(grid, a,b,c):
    i,j = grid
    if i < c.shape[0] and j < c.shape[1]:
        tmp = 0.
        for k in range(a.shape[1]): tmp += a[i, k] * b[k, j]
        c[i,j] = tmp

res = torch.zeros(m1.shape[0], m2.shape[1])
matmul((0,0), m1, m2, res)
print("============ a single operation ============")
print(res)
def launch_kernel(kernel, grid_x, grid_y, *args, **kwargs):
    for i in range(grid_x):
        for j in range(grid_y): kernel((i,j), *args, **kwargs)

launch_kernel(matmul, *res.shape, m1, m2, res)
print("============ all (parallelizable) operations============")
print(res)

### Add parallelization

In [ ]:
from numba import cuda

# here we don't need to pass `grid`. cuda runtime provides it.
@cuda.jit  # Compile this Python function into a CUDA GPU kernel (runs on the GPU).
def matmul(a, b, c):
    i, j = cuda.grid(2)  # Compute this thread's global 2D index (row=i, col=j) in the output.

    # Guard against threads whose (i, j) land outside the output matrix.
    if i < c.shape[0] and j < c.shape[1]:
        tmp = 0.0  # Accumulator for the dot product (float scalar).
        # Loop over shared dimension: a is (M,K), b is (K,N), so k goes 0..K-1
        for k in range(a.shape[1]):
            tmp += a[i, k] * b[k, j]  # One multiply-add for the dot product.
        c[i, j] = tmp  # Write this thread's result into output matrix c.

In [ ]:
r = np.zeros(tr.shape)  # Allocate host (CPU) output array with same shape as reference `tr`.

# Copy host arrays to GPU device memory: returns device arrays.
m1g, m2g, rg = map(cuda.to_device, (x_train, weights, r))

TPB = 16  # Threads Per Block along one dimension (so block is TPB x TPB threads).
rr, rc = r.shape  # Output rows and cols.
print(rr, rc)
# Compute grid size (blocks in x and y) needed to cover the output.
blockspergrid = (math.ceil(rr / TPB), math.ceil(rc / TPB))
print(blockspergrid)  # Show how many blocks you will launch in each grid dimension.

# Launch kernel:
#   grid = blockspergrid  (how many blocks total)
#   block = (TPB, TPB)    (threads in each block)
# In Numba CUDA syntax: kernel[gridDim, blockDim](args...)
matmul[blockspergrid, (TPB, TPB)](m1g, m2g, rg)
r = rg.copy_to_host()  # Copy GPU output back to CPU memory.
test_close(tr, r, eps=1e-3)  # Compare against reference with tolerance.

In [ ]:
%%timeit -n 50
matmul[blockspergrid, (TPB,TPB)](m1g,m2g,rg)
r = rg.copy_to_host()

### A simpler interface

In [ ]:
m1c,m2c = x_train.cuda(),weights.cuda()

In [ ]:
%timeit -n 50 r=(m1c@m2c).cpu()

- 12x faster than the einsum version
- 4200000x faster than the baseline